In [ ]:
import geopandas as gpd
from pathlib import Path
import fiona
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the output directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"


In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
# Path to your electricity_edges GeoPackage
electricity_gpkg_path = networks_folder / "energy/electricity_network_v3.1.gpkg"

layers = fiona.listlayers(electricity_gpkg_path)
print("Available layers:", layers)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
# Read the roads layers (edges and nodes) from the GeoPackage.
electricity_edges = gpd.read_file(electricity_gpkg_path, layer="edges")
electricity_nodes = gpd.read_file(electricity_gpkg_path, layer="nodes")

electricity_edges = electricity_edges.to_crs(jamaica_metric_grid_crs)
electricity_nodes = electricity_nodes.to_crs(jamaica_metric_grid_crs)

In [ ]:
electricity_edges.columns

In [ ]:
electricity_nodes.columns

In [ ]:
# === For Line Features (Road Edges) ===
# Perform an overlay (intersection) between the road edges and hydrobasins.
# This will split the roads by the hydrobasin boundaries.
electricity_edges_overlay = gpd.overlay(electricity_edges, hydrobasins, how="intersection")

# Optionally, calculate the length of each road segment (assuming a projected CRS)
electricity_edges_overlay["length"] = electricity_edges_overlay.geometry.length

# Example aggregation: Sum of road lengths by hydrobasin catchment (using HYBAS_ID)
electricity_length_by_catchment = electricity_edges_overlay.groupby("HYBAS_ID")["length"].sum().reset_index()
display("Electricity network Length by Catchment:")
display(electricity_length_by_catchment)

In [ ]:
# === For Point Features (Road Nodes) ===
# Perform a spatial join to attach hydrobasin attributes (e.g., HYBAS_ID) to each node.
electricity_nodes_join = gpd.sjoin(electricity_nodes, hydrobasins, how="left", predicate="intersects")

display(electricity_nodes_join.head())
display(electricity_nodes_join.columns)

# Example aggregation: Count the number of nodes per catchment
electricity_nodes_join_count_by_catchment = electricity_nodes_join.groupby("HYBAS_ID").size().reset_index(name="node_count")
display("\nNode Count by Catchment:")
display(electricity_nodes_join_count_by_catchment)

In [ ]:
# Define the output file path for the road edges layer
electricity_edges_catchments_intersection = networks_catchments_intersections / "electricity_edges_catchments_intersection.gpkg"

# Save the intersected edges layer to the new GeoPackage file
electricity_edges_overlay.to_file(electricity_edges_catchments_intersection, layer="intersected_edges", driver="GPKG")

# Save the nodes join layer to a new GeoPackage
electricity_nodes_catchments_intersection = networks_catchments_intersections / "electricity_nodes_catchments_intersection.gpkg"
electricity_nodes_join.to_file(electricity_nodes_catchments_intersection, layer="joined_nodes", driver="GPKG")